# Step 1 – Vision: anomaly detection on MVTec AD

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ayu-98-kumari/Adaptive_Visual_Inspection-_System/blob/main/notebooks/step1_vision.ipynb)

Part of the team project **Adaptive Visual Inspection System with RL-Based Inspection Scheduling**
(vision → RL scheduling → NLP/RAG reports → BI dashboard). This notebook covers **only the vision part**.

**What it does**
1. Setup: install anything missing, config, mount Drive, check GPU, count images.
2. Preprocessing: resize to 224×224 + ImageNet normalisation (+ one optional classic step).
3. Features: a pretrained ResNet-18 (no training) turns every image into a 512-number vector.
4. Anomaly detection: three simple methods score how "unusual" each test image is.
5. Evaluation: AUC, confusion matrix, recall and false-alarm rate per category and method.
6. *(Optional)* Defect-type classification with logistic regression.
7. Save outputs for teammates: `anomaly_scores.csv`, `metrics.json` and plots.

**How to run:** Runtime → Run all. Drive will ask for permission once.
All settings live in the **config cell** below; you should not need to edit anything else.

## 1. Setup

### 1.1 Install missing packages
Colab already has everything we need (torch, torchvision, scikit-learn, OpenCV, …).
This cell only installs a package if it is missing, so it normally does nothing.

In [ ]:
import importlib.util
import subprocess
import sys

# import name -> pip package name
REQUIRED = {
    "torch": "torch",
    "torchvision": "torchvision",
    "sklearn": "scikit-learn",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "cv2": "opencv-python-headless",
    "PIL": "Pillow",
}
missing = [pkg for module, pkg in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already installed.")

### 1.2 Config (the only cell you should need to edit)
- In **Colab**, data is read from and outputs are written to your Google Drive.
- Outside Colab (e.g. a quick logic check on a laptop), paths switch to `LOCAL_ROOT`,
  which can hold a tiny dummy dataset with the same folder layout.
- `PREPROCESS_STEP` turns the optional classic preprocessing step on (`"hist_eq"` or `"blur"`) or off (`None`).
- `RUN_DEFECT_CLASSIFIER` turns the optional Section 6 on or off.

In [ ]:
# ======================= CONFIG =======================
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# --- Paths ---
DRIVE_ROOT = "/content/drive/MyDrive/Integrated"  # used in Colab
LOCAL_ROOT = "../local_test"                      # used outside Colab (dummy data, git-ignored)
PROJECT_ROOT = DRIVE_ROOT if IN_COLAB else LOCAL_ROOT

DATA_DIR = f"{PROJECT_ROOT}/data/mvtec"
OUTPUT_DIR = f"{PROJECT_ROOT}/outputs/vision"
FEATURE_DIR = f"{OUTPUT_DIR}/features"  # cached embeddings (.npy)
PLOT_DIR = f"{OUTPUT_DIR}/plots"

# --- Data ---
CATEGORIES = ["grid", "zipper", "hazelnut", "leather", "toothbrush"]

# --- Preprocessing ---
IMAGE_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
PREPROCESS_STEP = None      # None, "hist_eq" (histogram equalisation) or "blur" (Gaussian blur)
BLUR_KERNEL_SIZE = 5        # only used when PREPROCESS_STEP == "blur" (must be odd)

# --- Feature extraction ---
BATCH_SIZE = 32
NUM_WORKERS = 2 if IN_COLAB else 0
USE_FEATURE_CACHE = True    # reuse saved embeddings from Drive if they exist

# --- Anomaly detection ---
KNN_K = 5                   # neighbours for kNN
OCSVM_NU = 0.1              # One-Class SVM: expected share of outliers in training data
NORMALIZE_CV_FOLDS = 5      # folds used to score good training images for the 0-1 normalised score

# --- Optional defect-type classifier (Section 6) ---
RUN_DEFECT_CLASSIFIER = True

RANDOM_SEED = 42
# ======================================================

print("Running in Colab:", IN_COLAB)
print("DATA_DIR:  ", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

### 1.3 Mount Drive, imports and device check
Colab sessions reset, so everything we save goes to Google Drive.
We use the GPU if one is available (Runtime → Change runtime type → GPU), otherwise the CPU.
The CPU also works, it is just slower for feature extraction.

In [ ]:
import os
import json
import random

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.covariance import LedoitWolf
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, confusion_matrix,
                             precision_recall_curve, roc_auc_score, roc_curve)
from sklearn.model_selection import KFold, train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

assert os.path.isdir(DATA_DIR), f"Data folder not found: {DATA_DIR}"
missing_categories = [c for c in CATEGORIES if not os.path.isdir(os.path.join(DATA_DIR, c))]
assert not missing_categories, f"Missing category folders in {DATA_DIR}: {missing_categories}"

for folder in [OUTPUT_DIR, FEATURE_DIR, PLOT_DIR]:
    os.makedirs(folder, exist_ok=True)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, f"({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else "")

### 1.4 List all images and count them
We walk the standard MVTec AD folders (`train/good`, `test/good`, `test/<defect_type>`) and build one table.
The folder name is the true defect type; `label` is 0 for good and 1 for defective.
The `ground_truth/` masks are not used.
Image paths are stored **relative to `DATA_DIR`** (e.g. `grid/test/bent/000.png`) so they work on any machine.

In [ ]:
IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg", ".bmp")

def list_images(category):
    rows = []
    for split in ["train", "test"]:
        split_dir = os.path.join(DATA_DIR, category, split)
        for defect_type in sorted(os.listdir(split_dir)):
            folder = os.path.join(split_dir, defect_type)
            if not os.path.isdir(folder):
                continue
            for file_name in sorted(os.listdir(folder)):
                if file_name.lower().endswith(IMAGE_EXTENSIONS):
                    rows.append({
                        "category": category,
                        "split": split,
                        "image_path": f"{category}/{split}/{defect_type}/{file_name}",
                        "true_defect_type": defect_type,
                        "label": int(defect_type != "good"),
                    })
    return rows

image_df = pd.DataFrame([row for c in CATEGORIES for row in list_images(c)])

counts = image_df.groupby("category").apply(lambda d: pd.Series({
    "train_good": ((d.split == "train") & (d.label == 0)).sum(),
    "test_good": ((d.split == "test") & (d.label == 0)).sum(),
    "test_defective": ((d.split == "test") & (d.label == 1)).sum(),
    "defect_types": d.loc[d.label == 1, "true_defect_type"].nunique(),
})).loc[CATEGORIES]
display(counts)
print("Total images:", len(image_df))

## 2. Preprocessing
Every image is resized to **224×224** and normalised with the **ImageNet mean/std**,
because that is what the pretrained ResNet-18 expects.

Optionally, one classic image-processing step is applied after resizing (set `PREPROCESS_STEP` in the config):
- `"hist_eq"` – histogram equalisation: spreads out brightness so contrast is more even.
- `"blur"` – Gaussian blur: smooths sensor noise and tiny texture details.

Below are a few before/after examples (one defective image per category).
If the step is off, we preview histogram equalisation just to show what it would do.

In [ ]:
def classic_step(img, step):
    """Optional classic preprocessing on an RGB uint8 image."""
    if step == "hist_eq":
        # Equalise only the brightness channel so colours are not distorted
        ycrcb = cv2.cvtColor(img, cv2.COLOR_RGB2YCrCb)
        ycrcb[:, :, 0] = cv2.equalizeHist(ycrcb[:, :, 0])
        return cv2.cvtColor(ycrcb, cv2.COLOR_YCrCb2RGB)
    if step == "blur":
        return cv2.GaussianBlur(img, (BLUR_KERNEL_SIZE, BLUR_KERNEL_SIZE), 0)
    return img

def load_image(rel_path, step=PREPROCESS_STEP):
    """Read an image, convert to RGB, resize to IMAGE_SIZE and apply the optional classic step."""
    img = np.array(Image.open(os.path.join(DATA_DIR, rel_path)).convert("RGB"))
    img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)
    return classic_step(img, step)

# Converts the uint8 image to a float tensor and applies ImageNet normalisation
to_tensor = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

preview_step = PREPROCESS_STEP or "hist_eq"
examples = image_df[image_df.label == 1].groupby("category").head(1).set_index("category").loc[CATEGORIES]

fig, axes = plt.subplots(2, len(CATEGORIES), figsize=(3 * len(CATEGORIES), 6), squeeze=False)
for i, (category, row) in enumerate(examples.iterrows()):
    axes[0, i].imshow(load_image(row.image_path, step=None))
    axes[0, i].set_title(f"{category}\n({row.true_defect_type})")
    axes[1, i].imshow(load_image(row.image_path, step=preview_step))
    axes[1, i].set_title(f"after {preview_step}")
for ax in axes.ravel():
    ax.axis("off")
plt.suptitle(f"Before / after preprocessing (active step: {PREPROCESS_STEP})")
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/preprocessing_examples.png", dpi=120)
plt.show()

## 3. Feature extraction with a pretrained ResNet-18
We use ResNet-18 pretrained on ImageNet **without any training**. We remove its final
classification layer, so instead of predicting ImageNet classes it outputs a **512-number vector
(embedding)** that describes what the image looks like. Similar-looking images get similar vectors.

Embeddings are cached to Drive as `.npy` files, so a re-run skips this slow step.
The cache is reused only if the list of images is unchanged (and is stored per preprocessing setting).

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, rel_paths):
        self.rel_paths = rel_paths

    def __len__(self):
        return len(self.rel_paths)

    def __getitem__(self, idx):
        return to_tensor(load_image(self.rel_paths[idx]))

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = torch.nn.Identity()   # drop the final layer -> 512-d output
model = model.eval().to(DEVICE)

@torch.no_grad()
def extract_features(rel_paths):
    loader = DataLoader(ImageDataset(rel_paths), batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=NUM_WORKERS)
    return np.concatenate([model(batch.to(DEVICE)).cpu().numpy() for batch in loader])

step_tag = PREPROCESS_STEP or "none"
features = {}  # (category, split) -> array of shape (n_images, 512), same order as image_df
for category in CATEGORIES:
    for split in ["train", "test"]:
        paths = image_df[(image_df.category == category) & (image_df.split == split)].image_path.tolist()
        npy_file = f"{FEATURE_DIR}/{category}_{split}_{step_tag}.npy"
        paths_file = f"{FEATURE_DIR}/{category}_{split}_{step_tag}_paths.txt"

        cache_ok = (USE_FEATURE_CACHE and os.path.exists(npy_file) and os.path.exists(paths_file)
                    and open(paths_file).read().splitlines() == paths)
        if cache_ok:
            feats = np.load(npy_file)
            print(f"{category:>10} {split:>5}: loaded {feats.shape} from cache")
        else:
            feats = extract_features(paths)
            np.save(npy_file, feats)
            with open(paths_file, "w") as f:
                f.write("\n".join(paths))
            print(f"{category:>10} {split:>5}: extracted {feats.shape}")
        features[(category, split)] = feats

## 4. Anomaly detection (good vs defective)
The training set contains **only good images**. Each method learns what "good" looks like and gives
every test image an **anomaly score** (higher = more unusual = more likely defective).
All three methods use the same embeddings:

| Method | Idea |
|---|---|
| **kNN (cosine, k=5)** | Average cosine distance to the 5 most similar good training images. |
| **Mahalanobis** | Distance from the centre of the good images, taking their spread into account (shrunk covariance, because we have fewer images than dimensions). |
| **One-Class SVM** | Learns a boundary around the good images; points outside get high scores. |

**Threshold:** to turn a score into a yes/no decision we need a threshold. For each category and method
we pick the threshold that gives the best F1 score **on the test set**.
⚠️ This is a simplification: it uses test labels, so results are optimistic. In a real system the threshold
would be chosen on a separate validation set (or set by the RL scheduler / business rules).

**Normalised score (0–1):** raw scores are on very different scales (kNN ≈ 0–0.5, Mahalanobis in the tens,
One-Class SVM can be negative), so they are hard to compare. We therefore also report a `normalized_score`:
the **share of good training images that look less unusual than this image**. For example, 0.97 means
"more unusual than 97% of good parts", and 1.0 means more unusual than all of them. It means the same thing for
every method. To score the good training images fairly, each one is scored by a model that did not see it
(K-fold). The normalised score is for display and comparison only. AUC, metrics and `prediction` are based on the
raw score.

In [ ]:
def knn_scores(train, test):
    nn_model = NearestNeighbors(n_neighbors=KNN_K, metric="cosine").fit(train)
    distances, _ = nn_model.kneighbors(test)
    return distances.mean(axis=1)

def mahalanobis_scores(train, test):
    cov = LedoitWolf().fit(train)
    return np.sqrt(cov.mahalanobis(test))  # .mahalanobis returns squared distances

def ocsvm_scores(train, test):
    scaler = StandardScaler().fit(train)
    svm = OneClassSVM(kernel="rbf", gamma="scale", nu=OCSVM_NU).fit(scaler.transform(train))
    return -svm.decision_function(scaler.transform(test))  # flip sign: higher = more anomalous

METHODS = {
    "knn_cosine": knn_scores,
    "mahalanobis": mahalanobis_scores,
    "ocsvm": ocsvm_scores,
}

def best_f1_threshold(labels, scores):
    """Threshold that maximises F1 on the given labels (simplification: uses test labels)."""
    precision, recall, thresholds = precision_recall_curve(labels, scores)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)
    return float(thresholds[np.argmax(f1[:-1])])

def cross_val_train_scores(score_fn, train):
    """Score every good training image with a model that did not see it (K-fold)."""
    scores = np.zeros(len(train))
    folds = KFold(n_splits=NORMALIZE_CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    for fit_idx, held_idx in folds.split(train):
        scores[held_idx] = score_fn(train[fit_idx], train[held_idx])
    return scores

def to_percentile(values, good_scores):
    """Share of good-image scores that are lower than each value -> 0..1."""
    good_scores = np.sort(good_scores)
    return np.searchsorted(good_scores, values, side="left") / len(good_scores)

test_df = image_df[image_df.split == "test"].reset_index(drop=True)
score_rows = []   # one row per (image, method) -> becomes anomaly_scores.csv
metrics = {}      # category -> method -> numbers -> becomes metrics.json
for category in CATEGORIES:
    cat_test = test_df[test_df.category == category]
    labels = cat_test.label.values
    metrics[category] = {}
    for method, score_fn in METHODS.items():
        scores = score_fn(features[(category, "train")], features[(category, "test")])
        threshold = best_f1_threshold(labels, scores)
        preds = (scores >= threshold).astype(int)

        good_train_scores = cross_val_train_scores(score_fn, features[(category, "train")])
        norm_scores = to_percentile(scores, good_train_scores)
        norm_threshold = float(to_percentile(threshold, good_train_scores))

        tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
        metrics[category][method] = {
            "auc": float(roc_auc_score(labels, scores)),
            "recall": float(tp / (tp + fn)),
            "false_alarm_rate": float(fp / (fp + tn)),
            "f1": float(2 * tp / (2 * tp + fp + fn)),
            "threshold": threshold,
            "normalized_threshold": norm_threshold,
            "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
        }
        for (_, row), score, norm_score, pred in zip(cat_test.iterrows(), scores, norm_scores, preds):
            score_rows.append({
                "category": category,
                "image_path": row.image_path,
                "true_defect_type": row.true_defect_type,
                "label": row.label,
                "method": method,
                "anomaly_score": float(score),
                "threshold": threshold,
                "prediction": int(pred),
                "normalized_score": float(norm_score),
                "normalized_threshold": norm_threshold,
            })

scores_df = pd.DataFrame(score_rows)

# Filled by the optional Section 6. If that section is removed, these stay empty and
# predicted_defect_type falls back to the true folder name.
predicted_defect_types = {}   # image_path -> predicted defect type
defect_metrics = {}           # category -> accuracy etc.

print(f"Scored {len(test_df)} test images with {len(METHODS)} methods -> {len(scores_df)} rows")

## 5. Evaluation
For each category and method:
- **AUC** – how well the score separates good from defective, independent of the threshold (1.0 = perfect, 0.5 = random).
- **Recall** – share of defective images we catch (missed defects are expensive).
- **False-alarm rate** – share of good images wrongly flagged as defective (causes unnecessary re-inspections).
- **Confusion matrix** – counts of correct and wrong decisions at the chosen threshold.

The RL teammate uses recall and false-alarm rate to model how reliable an inspection is.
The best method (highest AUC) is highlighted in green.

In [ ]:
summary_df = pd.DataFrame([
    {"category": c, "method": m, **{k: metrics[c][m][k] for k in ["auc", "recall", "false_alarm_rate", "f1", "threshold"]}}
    for c in CATEGORIES for m in METHODS
])

auc_table = summary_df.pivot(index="category", columns="method", values="auc").loc[CATEGORIES, list(METHODS)]
auc_table.loc["MEAN"] = auc_table.mean()
best_method = auc_table.loc["MEAN"].idxmax()

print("AUC per category and method (best per row in green):")
display(auc_table.style.highlight_max(axis=1, color="lightgreen").format("{:.3f}"))

print("Full summary:")
display(summary_df.style.format({k: "{:.3f}" for k in ["auc", "recall", "false_alarm_rate", "f1", "threshold"]}))

print(f"Best method overall (mean AUC): {best_method} ({auc_table.loc['MEAN', best_method]:.3f})")
for c in CATEGORIES:
    print(f"  best for {c:>10}: {auc_table.loc[c].idxmax()}")

In [ ]:
# ROC curves: one panel per category, one line per method
fig, axes = plt.subplots(1, len(CATEGORIES), figsize=(4 * len(CATEGORIES), 4), squeeze=False)
for ax, category in zip(axes[0], CATEGORIES):
    cat_scores = scores_df[scores_df.category == category]
    for method in METHODS:
        m = cat_scores[cat_scores.method == method]
        fpr, tpr, _ = roc_curve(m.label, m.anomaly_score)
        ax.plot(fpr, tpr, label=f"{method} (AUC={metrics[category][method]['auc']:.2f})")
    ax.plot([0, 1], [0, 1], "k--", linewidth=0.8)
    ax.set_title(category)
    ax.set_xlabel("False-alarm rate")
    ax.set_ylabel("Recall")
    ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/roc_curves.png", dpi=120)
plt.show()

# Confusion matrices: rows = categories, columns = methods
fig, axes = plt.subplots(len(CATEGORIES), len(METHODS), figsize=(3.2 * len(METHODS), 3 * len(CATEGORIES)), squeeze=False)
for i, category in enumerate(CATEGORIES):
    for j, method in enumerate(METHODS):
        r = metrics[category][method]
        cm = np.array([[r["tn"], r["fp"]], [r["fn"], r["tp"]]])
        ConfusionMatrixDisplay(cm, display_labels=["good", "defect"]).plot(ax=axes[i, j], colorbar=False)
        axes[i, j].set_title(f"{category} – {method}", fontsize=9)
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/confusion_matrices.png", dpi=120)
plt.show()

### Score distributions on the common 0–1 scale
For every category and method: how the normalised scores of **good** (green) and **defective** (red) test images
are spread, with the threshold as a dashed line. The less the two colours overlap, the better the method separates
good from defective. Because all methods share the same 0–1 axis, they can be compared side by side.

In [ ]:
bins = np.linspace(0, 1, 21)
fig, axes = plt.subplots(len(CATEGORIES), len(METHODS), figsize=(3.6 * len(METHODS), 2.6 * len(CATEGORIES)),
                         squeeze=False, sharex=True)
for i, category in enumerate(CATEGORIES):
    for j, method in enumerate(METHODS):
        ax = axes[i, j]
        m = scores_df[(scores_df.category == category) & (scores_df.method == method)]
        ax.hist(m[m.label == 0].normalized_score, bins=bins, alpha=0.6, color="tab:green", label="good")
        ax.hist(m[m.label == 1].normalized_score, bins=bins, alpha=0.6, color="tab:red", label="defective")
        ax.axvline(metrics[category][method]["normalized_threshold"], color="black", linestyle="--", label="threshold")
        ax.set_title(f"{category} – {method}", fontsize=9)
axes[0, 0].legend(fontsize=8)
for ax in axes[-1]:
    ax.set_xlabel("Normalised score")
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/score_distributions.png", dpi=120)
plt.show()

### Which method do we continue with, and why?
We compare the methods on:
1. **Overall quality**: mean AUC over all categories (does not depend on the threshold). This is the selection rule.
2. **Consistency**: in how many categories the method has the best AUC.
3. **What matters in inspection**: mean recall (defects caught) and mean false-alarm rate at the chosen thresholds.

The cell below fills in the result automatically and saves it to `method_choice.json`.
Remember: thresholds, and therefore this choice, are based on the test set (a prototype simplification).

In [ ]:
best_per_category = auc_table.drop("MEAN").idxmax(axis=1)

method_summary = summary_df.groupby("method")[["auc", "recall", "false_alarm_rate"]].mean().loc[list(METHODS)]
method_summary["categories_won"] = [int((best_per_category == m).sum()) for m in METHODS]
display(method_summary.style.format({"auc": "{:.3f}", "recall": "{:.1%}", "false_alarm_rate": "{:.1%}"})
        .highlight_max(subset=["auc"], color="lightgreen"))

chosen_method = best_method
chosen = method_summary.loc[chosen_method]
print(f"Chosen method: {chosen_method}")
print(f"  - highest mean AUC: {chosen.auc:.3f}")
print(f"  - best AUC in {int(chosen.categories_won)} of {len(CATEGORIES)} categories")
print(f"  - mean recall {chosen.recall:.1%} at a mean false-alarm rate of {chosen.false_alarm_rate:.1%}")

## 6. (Optional) Defect-type classification
*This whole section can be deleted (or switched off with `RUN_DEFECT_CLASSIFIER = False`).
The rest of the notebook still works; `predicted_defect_type` then simply equals the true folder name.*

Using **only the defective test images**, we try to predict *which kind* of defect it is (the folder name,
e.g. `crack`, `hole`). Per category we split these images 50/50 (stratified, so every defect type is in
both halves), train a logistic regression on the embeddings of one half and test on the other.
Accuracy and the confusion matrix are reported on the held-out half. The results table also shows what
**random guessing** would give (1 ÷ number of defect types), so you can see how much better than chance the
classifier is.

To give **every** defective image a prediction from a model that never saw it, we then also train on the
second half and predict the first half. These out-of-sample predictions go into `anomaly_scores.csv`.
Toothbrush is skipped because it has only one defect type.

In [ ]:
if RUN_DEFECT_CLASSIFIER:
    def train_defect_classifier(X, y):
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
        return clf.fit(X, y)

    eval_results = {}  # category -> (y_true, y_pred) on the held-out half, for plotting
    table_rows = []    # one row per category for the results table
    for category in CATEGORIES:
        is_cat = (test_df.category == category).values
        cat_test = test_df[is_cat].reset_index(drop=True)
        defective = (cat_test.label == 1).values
        X = features[(category, "test")][defective]
        y = cat_test.true_defect_type.values[defective]
        paths = cat_test.image_path.values[defective]

        n_types = len(np.unique(y))
        if n_types < 2:
            table_rows.append({"Category": category, "Accuracy": np.nan, "Defect types": n_types,
                               "Random guessing would give": np.nan, "Test images": np.nan})
            continue

        idx_a, idx_b = train_test_split(np.arange(len(y)), test_size=0.5, stratify=y, random_state=RANDOM_SEED)

        # Main evaluation: train on half A, test on half B
        pred_b = train_defect_classifier(X[idx_a], y[idx_a]).predict(X[idx_b])
        # Swap halves so half A also gets an out-of-sample prediction
        pred_a = train_defect_classifier(X[idx_b], y[idx_b]).predict(X[idx_a])

        for i, p in zip(idx_b, pred_b):
            predicted_defect_types[paths[i]] = p
        for i, p in zip(idx_a, pred_a):
            predicted_defect_types[paths[i]] = p

        acc = accuracy_score(y[idx_b], pred_b)
        defect_metrics[category] = {
            "accuracy": float(acc),
            "random_guess_accuracy": 1 / n_types,
            "n_defect_types": n_types,
            "n_train": int(len(idx_a)),
            "n_test": int(len(idx_b)),
        }
        eval_results[category] = (y[idx_b], pred_b)
        table_rows.append({"Category": category, "Accuracy": acc, "Defect types": n_types,
                           "Random guessing would give": 1 / n_types, "Test images": len(idx_b)})

    # Results table: accuracy on the held-out half vs. what random guessing would give
    defect_table = pd.DataFrame(table_rows).set_index("Category")
    display(defect_table.style.format({
        "Accuracy": lambda v: "skipped (only one defect type)" if pd.isna(v) else f"{v:.2f}",
        "Random guessing would give": lambda v: "–" if pd.isna(v) else f"{v:.2f}",
        "Test images": lambda v: "–" if pd.isna(v) else f"{v:.0f}",
    }))

    if eval_results:
        fig, axes = plt.subplots(1, len(eval_results), figsize=(5 * len(eval_results), 5), squeeze=False)
        for ax, (category, (y_true, y_pred)) in zip(axes[0], eval_results.items()):
            ConfusionMatrixDisplay.from_predictions(y_true, y_pred, ax=ax, colorbar=False, xticks_rotation=45)
            ax.set_title(f"{category} (acc={defect_metrics[category]['accuracy']:.2f})")
        plt.tight_layout()
        plt.savefig(f"{PLOT_DIR}/defect_type_confusion.png", dpi=120)
        plt.show()
else:
    print("Defect-type classifier switched off (RUN_DEFECT_CLASSIFIER = False).")

## 7. Save outputs for teammates
Everything goes to `OUTPUT_DIR` on Drive:

| File | Contents | Used by |
|---|---|---|
| `anomaly_scores.csv` | One row per test image **and** method: `category, image_path, true_defect_type, label, method, anomaly_score, threshold, prediction, predicted_defect_type, normalized_score, normalized_threshold` | RL, NLP/RAG, BI |
| `metrics.json` | Per category and method: `auc, recall, false_alarm_rate, threshold` (+ f1, normalized_threshold and confusion counts) | RL (recall & false-alarm rate), BI |
| `summary_table.csv` | The summary table from Section 5 | BI / report |
| `method_choice.json` | The chosen method, the selection rule, and mean AUC / recall / false-alarm rate per method | report, BI |
| `defect_type_metrics.json` / `defect_type_summary.csv` | Accuracy of the optional defect-type classifier vs. random guessing (only if Section 6 ran) | NLP/RAG, report |
| `plots/*.png` | Preprocessing examples, ROC curves, confusion matrices, score distributions | report |

Notes: `label` is 0 = good, 1 = defective; `prediction` uses the same coding.
`normalized_score` is on a common 0–1 scale for all methods (share of good training images that are less unusual);
`prediction` is based on the raw `anomaly_score` and `threshold`.
`image_path` is relative to the MVTec data folder. `predicted_defect_type` comes from Section 6 when it ran,
otherwise it is the true folder name (`good` for good images).

In [ ]:
scores_df["predicted_defect_type"] = [
    predicted_defect_types.get(path, true_type)
    for path, true_type in zip(scores_df.image_path, scores_df.true_defect_type)
]
columns = ["category", "image_path", "true_defect_type", "label", "method",
           "anomaly_score", "threshold", "prediction", "predicted_defect_type",
           "normalized_score", "normalized_threshold"]
scores_df[columns].to_csv(f"{OUTPUT_DIR}/anomaly_scores.csv", index=False)

with open(f"{OUTPUT_DIR}/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

summary_df.to_csv(f"{OUTPUT_DIR}/summary_table.csv", index=False)

method_choice = {
    "chosen_method": chosen_method,
    "selection_rule": "highest mean AUC over all categories (thresholds tuned on the test set)",
    "best_method_per_category": best_per_category.to_dict(),
    "per_method": {
        m: {"mean_auc": float(r.auc), "mean_recall": float(r.recall),
            "mean_false_alarm_rate": float(r.false_alarm_rate), "categories_won": int(r.categories_won)}
        for m, r in method_summary.iterrows()
    },
}
with open(f"{OUTPUT_DIR}/method_choice.json", "w") as f:
    json.dump(method_choice, f, indent=2)

if defect_metrics:
    with open(f"{OUTPUT_DIR}/defect_type_metrics.json", "w") as f:
        json.dump(defect_metrics, f, indent=2)
    pd.DataFrame.from_dict(defect_metrics, orient="index").rename_axis("category").to_csv(
        f"{OUTPUT_DIR}/defect_type_summary.csv")

print("Saved files in", OUTPUT_DIR)
for root, _, files in os.walk(OUTPUT_DIR):
    if root.startswith(FEATURE_DIR):
        continue
    for name in sorted(files):
        print("  ", os.path.relpath(os.path.join(root, name), OUTPUT_DIR))
print("\nPreview of anomaly_scores.csv:")
display(scores_df[columns].head())